
# Fast COPER demo on local MIMIC-III

This notebook shows a **minimal and reproducible pipeline**:
1. load a local MIMIC-III sample,
2. compute a few quick indicators,
3. build a compact time-series tensor,
4. run a quick **COPER** training demo on a binary task (in-hospital mortality).

> Practical goal: a demonstration run in **< 2 minutes** (sampled data + small model).


In [1]:
from pathlib import Path
import sys
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# Make the src package importable from anywhere (Jupyter cwd can vary)

def _find_repo_root_with_src(start: Path) -> Path:
    p = start.resolve()
    for _ in range(12):
        if (p / "src").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent

    # Common case: we're in `.../notebooks/`
    if (start.parent / "src").is_dir():
        return start.parent

    raise RuntimeError(f"Could not find repo root containing 'src/': start={start}")


PROJECT_ROOT = _find_repo_root_with_src(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.coper_model import COPER
from src.load_dataset import Dataset

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

MIMIC_DIR = Path('/home/charlesv/Desktop/StatisitcalGenetics/MIMIC/physionet.org/files/mimiciii/1.4')
assert MIMIC_DIR.exists(), f"MIMIC folder not found: {MIMIC_DIR}"

Device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU



## 1) Local MIMIC loading and quick indicators

We read only a **subset** of the tables to keep runtime short.

Displayed indicators:
- sample size,
- in-hospital mortality prevalence,
- number of unique patients/admissions,
- most frequent clinical variables (`ITEMID`) observed in `CHARTEVENTS`.


In [ ]:
# "Fast demo" parameters
N_ROWS_ADM = 80_000
N_ROWS_CHART = 320_000
N_FEATURES = 8
MAX_PATIENTS = 220
SEQ_LEN = 48   # 48 time steps (48-hour window, hourly bins)

admissions = pd.read_csv(
    MIMIC_DIR / 'ADMISSIONS.csv.gz',
    compression='gzip',
    usecols=['HADM_ID', 'SUBJECT_ID', 'HOSPITAL_EXPIRE_FLAG'],
    nrows=N_ROWS_ADM,
)

chart = pd.read_csv(
    MIMIC_DIR / 'CHARTEVENTS.csv.gz',
    compression='gzip',
    usecols=['HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM'],
    nrows=N_ROWS_CHART,
)

chart = chart.dropna(subset=['HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM']).copy()
chart['HADM_ID'] = chart['HADM_ID'].astype(np.int64)
chart['ITEMID'] = chart['ITEMID'].astype(np.int64)
chart['CHARTTIME'] = pd.to_datetime(chart['CHARTTIME'], errors='coerce')
chart = chart.dropna(subset=['CHARTTIME'])

# Restrict to admissions loaded above
chart = chart[chart['HADM_ID'].isin(admissions['HADM_ID'])].copy()

# Keep the most frequent variables
top_itemids = chart['ITEMID'].value_counts().head(N_FEATURES).index.to_list()
chart = chart[chart['ITEMID'].isin(top_itemids)].copy()

# A few global indicators
mortality_rate = admissions['HOSPITAL_EXPIRE_FLAG'].mean()
print('Admissions (sample):', len(admissions))
print('Unique subjects:', admissions['SUBJECT_ID'].nunique())
print('In-hospital mortality prevalence:', round(float(mortality_rate), 4))
print('CHARTEVENTS rows kept:', len(chart))
print('Top ITEMID:', top_itemids)

# Make ITEMIDs human-readable using MIMIC's D_ITEMS.csv.gz
# (columns include ITEMID + LABEL)
items_path = MIMIC_DIR / 'D_ITEMS.csv.gz'
if items_path.is_file():
    d_items = pd.read_csv(
        items_path,
        compression='gzip',
        usecols=['ITEMID', 'LABEL'],
    )
    d_items['ITEMID'] = d_items['ITEMID'].astype(np.int64)
    d_items = d_items[d_items['ITEMID'].isin(top_itemids)].drop_duplicates('ITEMID')
    item_to_label = dict(zip(d_items['ITEMID'].tolist(), d_items['LABEL'].tolist()))

    print('\nTop ITEMID labels:')
    for item in top_itemids:
        print(f"  {int(item)}: {item_to_label.get(int(item), 'N/A')}")
else:
    print(f"\nWARNING: Missing MIMIC dictionary file: {items_path}")

Admissions (sample): 58976
Unique subjects: 46520
In-hospital mortality prevalence: 0.0993
CHARTEVENTS rows kept: 144472
Top ITEMID: [220045, 220210, 220277, 220181, 220179, 220180, 220052, 220050]

Top ITEMID labels:
  220045: Heart Rate
  220210: Respiratory Rate
  220277: O2 saturation pulseoxymetry
  220181: Non Invasive Blood Pressure mean
  220179: Non Invasive Blood Pressure systolic
  220180: Non Invasive Blood Pressure diastolic
  220052: Arterial Blood Pressure mean
  220050: Arterial Blood Pressure systolic



## 2) Building a compact time-series representation

We build a tensor `X` of shape `(N, T, F)` with:
- `N`: number of admissions,
- `T=48`: time steps (normalized hours),
- `F`: retained clinical variables (`ITEMID`).

### Formalization
For patient `i`, variable `f`, and time bin `t`:

\[
X_{i,t,f} = \text{mean observed value for } (i,f) \text{ in bin } t.
\]

Then we apply:
1. forward fill to reduce missingness,
2. final fill with `0`.


In [3]:
# Time anchoring: first measurement per admission
chart['t0'] = chart.groupby('HADM_ID')['CHARTTIME'].transform('min')
chart['delta_h'] = (chart['CHARTTIME'] - chart['t0']).dt.total_seconds() / 3600.0
chart = chart[(chart['delta_h'] >= 0) & (chart['delta_h'] < SEQ_LEN)].copy()
chart['t_bin'] = np.floor(chart['delta_h']).astype(int)

# Mean value in each hourly bin
agg = (
    chart.groupby(['HADM_ID', 't_bin', 'ITEMID'])['VALUENUM']
    .mean()
    .reset_index()
)

# Admissions with enough signal
counts = agg.groupby('HADM_ID').size().sort_values(ascending=False)
selected_hadm = counts.head(MAX_PATIENTS).index.to_list()

item_to_col = {item: j for j, item in enumerate(top_itemids)}
label_map = admissions.set_index('HADM_ID')['HOSPITAL_EXPIRE_FLAG'].to_dict()

X_list, y_list = [], []
for hadm_id in selected_hadm:
    block = agg[agg['HADM_ID'] == hadm_id]
    if block.empty or hadm_id not in label_map:
        continue

    arr = np.full((SEQ_LEN, len(top_itemids)), np.nan, dtype=np.float32)
    for row in block.itertuples(index=False):
        t = int(row.t_bin)
        j = item_to_col.get(int(row.ITEMID))
        if j is None or t < 0 or t >= SEQ_LEN:
            continue
        arr[t, j] = float(row.VALUENUM)

    # Forward fill per feature, then fill remaining NaNs with 0
    arr_df = pd.DataFrame(arr).ffill(axis=0).fillna(0.0)
    arr = arr_df.to_numpy(dtype=np.float32)

    X_list.append(arr)
    y_list.append(float(label_map[hadm_id]))

X = np.stack(X_list)
y = np.array(y_list, dtype=np.float32)

print('X shape:', X.shape, ' | y shape:', y.shape)
print('Mortality rate (final cohort):', round(float(y.mean()), 4))


X shape: (220, 48, 8)  | y shape: (220,)
Mortality rate (final cohort): 0.0818



## 3) COPER: compact mathematical reminder

COPER combines a continuous encoder (Neural ODE) with a Perceiver architecture.

### Latent-state ODE
Between two observation times, the latent state follows:
\[
\frac{d\mathbf{h}(t)}{dt} = f_\theta(\mathbf{h}(t), t),
\]
with a numerical solver (here via `torchdiffeq`) to propagate the state between irregular timestamps.

### Perceiver cross-attention
With learned latents `\mathbf{Z}` and encoded data `\mathbf{X}`:
\[
\text{Attn}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)V.
\]
COPER applies cross-attention first (latents \(\leftarrow\) data), then self-attention blocks over latents.

### Classification head
We use the last latent and a sigmoid projection:
\[
\hat{y} = \sigma(W\,z_{\text{last}} + b).
\]


In [4]:

# Split train/test
if len(np.unique(y)) < 2:
    raise RuntimeError('Final cohort has only one class; increase N_ROWS_* or MAX_PATIENTS.')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test_t = torch.tensor(y_test, dtype=torch.float32, device=device)

train_ds = Dataset(X_train_t, y_train_t, carry_frd=False, drop=None)
test_ds = Dataset(X_test_t, y_test_t, carry_frd=False, drop=None)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False)

# Minimal COPER config for the demo
cfg = SimpleNamespace(
    setting='Train',
    ode_dropout=0.10,
    att_dropout=0.10,
    ff_dropout=0.10,
    self_per_cross_attn=1,
    latent_heads=1,
    cross_heads=1,
    cross_dim_head=32,
    latent_dim_head=32,
)

model = COPER(
    config=cfg,
    n_labels=1,
    input_dim=X.shape[-1],
    num_latents=16,
    latent_dim=32,
    rec_layers=1,
    units=32,
    nonlinear=nn.Tanh,
    cont_in=True,
    cont_out=False,
    emb_dim=32,
    device=device,
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

def evaluate(loader):
    model.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for batch in loader:
            pred = model(batch['X'], batch['tp'], batch['tp'], batch['tp']).squeeze(-1)
            probs_all.append(pred.detach().cpu().numpy())
            y_all.append(batch['y'].detach().cpu().numpy())
    probs = np.concatenate(probs_all)
    y_true = np.concatenate(y_all)
    metrics = {}
    if len(np.unique(y_true)) > 1:
        metrics['AUROC'] = roc_auc_score(y_true, probs)
        metrics['AUPRC'] = average_precision_score(y_true, probs)
    metrics['MeanProb'] = float(np.mean(probs))
    return metrics

start = time.time()
max_batches = 12   # hard cap to keep runtime short
n_epochs =10

for epoch in range(1, n_epochs + 1):
    model.train()
    epoch_loss = 0.0
    for b_idx, batch in enumerate(train_loader):
        if b_idx >= max_batches:
            break
        opt.zero_grad()
        pred = model(batch['X'], batch['tp'], batch['tp'], batch['tp']).squeeze(-1)
        loss = criterion(pred, batch['y'])
        loss.backward()
        opt.step()
        epoch_loss += float(loss.item())

    val_metrics = evaluate(test_loader)
    print(f"Epoch {epoch} | loss={epoch_loss / max(1, min(len(train_loader), max_batches)):.4f} | metrics={val_metrics}")

elapsed = time.time() - start
print(f"Training+eval runtime: {elapsed:.2f} sec")

Epoch 1 | loss=0.3981 | metrics={'AUROC': 0.212, 'AUPRC': 0.0685527664210742, 'MeanProb': 0.04072894901037216}
Epoch 2 | loss=0.3200 | metrics={'AUROC': 0.33199999999999996, 'AUPRC': 0.12823685691770798, 'MeanProb': 0.04256473854184151}
Epoch 3 | loss=0.2594 | metrics={'AUROC': 0.384, 'AUPRC': 0.11812063260453938, 'MeanProb': 0.07460443675518036}
Epoch 4 | loss=0.2723 | metrics={'AUROC': 0.408, 'AUPRC': 0.11111160253575114, 'MeanProb': 0.08405639231204987}
Epoch 5 | loss=0.2583 | metrics={'AUROC': 0.48, 'AUPRC': 0.11700123342584026, 'MeanProb': 0.06525994092226028}
Epoch 6 | loss=0.3342 | metrics={'AUROC': 0.592, 'AUPRC': 0.1483993993993994, 'MeanProb': 0.0662020891904831}
Epoch 7 | loss=0.2759 | metrics={'AUROC': 0.584, 'AUPRC': 0.14817317983195263, 'MeanProb': 0.09519557654857635}
Epoch 8 | loss=0.2609 | metrics={'AUROC': 0.74, 'AUPRC': 0.20065656565656564, 'MeanProb': 0.07810624688863754}
Epoch 9 | loss=0.2591 | metrics={'AUROC': 0.748, 'AUPRC': 0.21703222703222702, 'MeanProb': 0.06

## Save + reload the trained demo model

Cette section sauvegarde le modèle COPER entraîné, le recharge, puis produit des probabilités sur le `test_loader` (quelques prédictions affichées).

In [5]:
from pathlib import Path

# PROJECT_ROOT est défini en début de notebook
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = RESULTS_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = MODEL_DIR / f"demo_coper_state_seed{SEED}.pt"

# Snapshot minimal pour reconstruire le modèle
ckpt = {
    "model_state_dict": model.state_dict(),
    "cfg": vars(cfg),
    "input_dim": int(X.shape[-1]),
    "num_latents": 16,
    "latent_dim": 32,
    "rec_layers": 1,
    "units": 32,
    "emb_dim": 32,
    "nonlinear": "Tanh",
    "cont_in": True,
    "cont_out": False,
    "seed": SEED,
}

torch.save(ckpt, ckpt_path)

print("Saved model checkpoint to:", ckpt_path)
print("Checkpoint keys:", list(ckpt.keys()))

Saved model checkpoint to: /home/charlesv/Desktop/StatisitcalGenetics/code/COPER/results/models/demo_coper_state_seed2026.pt
Checkpoint keys: ['model_state_dict', 'cfg', 'input_dim', 'num_latents', 'latent_dim', 'rec_layers', 'units', 'emb_dim', 'nonlinear', 'cont_in', 'cont_out', 'seed']


In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# ---- Reload ----
reloaded_cfg = SimpleNamespace(**torch.load(ckpt_path, map_location="cpu")["cfg"])

ckpt = torch.load(ckpt_path, map_location="cpu")
reloaded_cfg = SimpleNamespace(**ckpt["cfg"])

reloaded_model = COPER(
    config=reloaded_cfg,
    n_labels=1,
    input_dim=ckpt["input_dim"],
    num_latents=ckpt["num_latents"],
    latent_dim=ckpt["latent_dim"],
    rec_layers=ckpt["rec_layers"],
    units=ckpt["units"],
    nonlinear=nn.Tanh,
    cont_in=ckpt["cont_in"],
    cont_out=ckpt["cont_out"],
    emb_dim=ckpt["emb_dim"],
    device=device,
).to(device)

reloaded_model.load_state_dict(ckpt["model_state_dict"], strict=True)
reloaded_model.eval()

# ---- Predict on test set ----

def predict_probs(model_, loader_):
    model_.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for batch in loader_:
            pred = model_(batch['X'], batch['tp'], batch['tp'], batch['tp']).squeeze(-1)
            probs_all.append(pred.detach().cpu().numpy())
            y_all.append(batch['y'].detach().cpu().numpy())
    probs = np.concatenate(probs_all)
    y_true = np.concatenate(y_all)
    return y_true, probs


y_true, probs = predict_probs(reloaded_model, test_loader)

if len(np.unique(y_true)) > 1:
    auroc = roc_auc_score(y_true, probs)
    auprc = average_precision_score(y_true, probs)
    print(f"Reloaded model metrics: AUROC={auroc:.4f} | AUPRC={auprc:.4f}")
else:
    print("Reloaded model metrics: only one class present in y_true; AUROC/AUPRC skipped")

# ---- Show a few predictions ----
N_SHOW = 12
# We reuse the first batch for easy inspection
first_batch = next(iter(test_loader))
with torch.no_grad():
    pred0 = reloaded_model(first_batch['X'][:N_SHOW], first_batch['tp'][:N_SHOW], first_batch['tp'][:N_SHOW], first_batch['tp'][:N_SHOW]).squeeze(-1)

probs0 = pred0.detach().cpu().numpy().reshape(-1)
y0 = first_batch['y'][:N_SHOW].detach().cpu().numpy().reshape(-1)

df = pd.DataFrame({
    "y_true": y0.astype(int),
    "prob_mortality": probs0,
})
df["y_pred"] = (df["prob_mortality"] >= 0.5).astype(int)

display(df)
print("Mean predicted probability:", float(np.mean(probs)))

Reloaded model metrics: AUROC=0.8040 | AUPRC=0.2406


,y_true,prob_mortality,y_pred
0,0,0.068287,0
1,0,0.067829,0
2,0,0.068705,0
3,0,0.069847,0
4,0,0.068185,0
5,1,0.068547,0
6,0,0.076457,0
7,0,0.068535,0
8,0,0.069112,0
9,0,0.067806,0


Mean predicted probability: 0.06870563328266144
